# Phase 03: Model Selection & Profit Optimization

That was initial planned: 

* **Algorithm Candidates:** Train high-performance classifiers (e.g., Random Forest, XGBoost) and establish a baseline using Logistic Regression.
* **Cost-Sensitive Learning:** Incorporate the 4:1 loss ratio ($100\%$ loss vs. $25\%$ margin) directly into the model training weights or cost function.
* **Threshold Optimization:** Instead of using the default 0.5 threshold, identify the specific probability cutoff (theoretically **0.20**) that maximizes the Profit Function.




# Setup and Imports


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SubPipeline

from src.etl import load_processed_features, load_features_descriptions,load_raw_mercadolibre_dataset
from src.models.model_selection import time_based_split
from src.models.evaluation import (
    get_scores,
    compute_pr_stats,
    compute_classification_report,
    print_classification_report,
    compute_profit_curve,
    compute_confusion_matrix_data,
    compute_fbeta_curve,
    plot_pr_curve,
    plot_profit_curve,
    plot_confusion_matrix,
    plot_fbeta_by_threshold,
)
from src.preprocessing.custom_transformers import (
    BinaryNaNTransformer,
    CategoricalEncoderWithNaN,
    MissingFlagTransformer,
    ZeroFlagTransformer,
    TopPercentileFlagTransformer,
    RareGroupTransformer,
    FrequencyEncodingTransformer,
    NamedPCATransformer,
)


In [2]:
df_raw = load_raw_mercadolibre_dataset()
df_features_descriptions = load_features_descriptions()


Loading raw MercadoLibre dataset from dataset\raw\MercadoLibre Data Scientist Technical Challenge - Dataset.csv...
Raw MercadoLibre dataset loaded successfully from dataset\raw\MercadoLibre Data Scientist Technical Challenge - Dataset.csv as a pandas DataFrame.
Features suggestion loaded from notebooks\results\phase_02_manual_features_descriptions.csv


# Business Objective & Financial Optimization

Our primary goal is to maximize the company's profit by effectively avoiding fraud.

Currently, we face a baseline fraud rate (e.g., `5%`), which results in a significant financial loss (`-$ 547,271.12`). In a naive baseline model where we simply approve all transactions, we absorb this full loss. Our objective is to build a model that decreases these fraudulent transactions and improves our bottom line.

In [3]:
n_total  = len(df_raw)
n_fraud  = df_raw['fraude'].sum()
fraud_rate = n_fraud / n_total

total_loss    = df_raw.loc[df_raw['fraude'] == 1, 'monto'].sum()
avg_fraud_amt = df_raw.loc[df_raw['fraude'] == 1, 'monto'].mean()

print(f"Total transactions : {n_total:>10,}")
print(f"Fraud transactions : {n_fraud:>10,}")
print(f"Fraud rate         : {fraud_rate:>10.2%}")
print()
print(f"Avg fraud amount   : {avg_fraud_amt:>10,.2f}")
print(f"Total fraud loss   : {total_loss:>10,.2f}  (100% of flagged monto)")
print(f"Baseline margin loss (approve all, 25% margin on legit blocked = 0): {total_loss:,.2f}")


Total transactions :    150,000
Fraud transactions :      7,500
Fraud rate         :      5.00%

Avg fraud amount   :      72.97
Total fraud loss   : 547,271.12  (100% of flagged monto)
Baseline margin loss (approve all, 25% margin on legit blocked = 0): 547,271.12


According with previous explanaiotn in notebook `00_strategic_modeling_framework.ipynb`, we define the Total Profit as the sum of the financial outcomes of each prediction:

$$M = \sum (TP \cdot 0) + \sum (TN \cdot 0) - \sum (FP \cdot 0.25M) - \sum (FN \cdot 1.0M)$$

$$M = - (1.0 \cdot FN) - (0.25 \cdot FP)$$

* **$M$**: The transaction amount (`monto`).
* **$TN$ (True Negative)**: Correctly approved legitimate transaction, but this is unaffected by the model's intervention.
* **$FP$ (False Positive)**: Legitimate transaction blocked. You lose the 25% margin you would have made (Opportunity Cost).
* **$FN$ (False Negative)**: Fraud approved. You lose 100% of the transaction value.
* **$TP$ (True Positive)**: Fraud blocked. The loss is avoided, but no new profit is created.

This will be a metric used to optmize model during the modeling. 

# Machine Learning Modeling for Fraud Detection: XGBoost

This section details the machine learning modeling phase. Below, we outline our methodology and the specific processes used for:

* **Data Preprocessing Pipeline:** Designing a robust processing architecture that ensures strict consistency between our experimental environment and our final production deployment.
* **Model Construction & Selection:** Justifying the choice of XGBoost as our primary algorithm for this highly imbalanced, cost-sensitive classification task.


## Data Processor

### Feature working

Thinking about **production** and keeping the feature engineering consistent with our Phase 2 analysis, we constructed a preprocessing **pipeline using scikit-learn**. The main goals are:

* **Reproducibility and consistency:** ensure the exact same transformations are applied in training, validation, test, and future production data.
* **Avoid data leakage:** transformations that need to “learn” parameters (e.g., bin thresholds, PCA components, encoding maps) are **fit only on the training split** and then reused to transform new data.
* **Robustness to missing values:** based on the previous analysis, the pipeline includes clear rules to handle **NaN** safely (so the model does not break and missingness can be used as signal when relevant).
* **Experiment vs production compatibility:** sklearn’s `fit()` / `transform()` pattern makes it easier to simulate production behavior during experiments and to store the fitted artifacts (encoders, PCA weights, etc.).

In [4]:
# Choose of feature was made based on the Phase 2 analysis, considering both feature importance and preliminary model performance. 
# The selected features include a mix of raw features, engineered flags, encoded categorical variables, 
# and PCA components to capture complex relationships in the data while maintaining interpretability and model efficiency.
# Feature set selected for XGBoost based on Phase 2 analysis.
XGB_FEATURES = [
    'a', 'b', 'd', 'e', 'f', 'g', 'l', 'm', 'n', 'o', 'p', 'monto', 'score',  # raw
    'o_is_missing', 'f_is_zero', 'm_is_zero', 'monto_top1_flag',                 # flags
    'j_rare_grouped',                                                             # encoded
    'pca_beh_1', 'pca_amt_1',                                                    # PCA
]

# Numeric columns passed through without transformation
passthrough_raw = ['a', 'b', 'd', 'e', 'f', 'l', 'm', 'n', 'monto', 'score']

preprocessor_xgb = ColumnTransformer(
    transformers=[
        ('missing_flag',        MissingFlagTransformer(),                              ['o']),
        ('zero_flag',           ZeroFlagTransformer(),                                 ['f', 'm']),
        ('top1_flag',           TopPercentileFlagTransformer(percentile=0.99),         ['monto']),
        ('binary_nan_handler',  BinaryNaNTransformer(),                                ['o', 'p']),
        ('categorical_encoder', CategoricalEncoderWithNaN(),                           ['g']),
        ('rare_grouper_j',      SubPipeline([
                                    ('rare', RareGroupTransformer(min_freq=0.01)),
                                    ('freq', FrequencyEncodingTransformer(output_suffix='')),
                                ]),                                                    ['j']),
        ('pca_beh',             NamedPCATransformer(n_components=1, prefix='pca_beh'), ['d', 'm']),
        ('pca_amt',             NamedPCATransformer(n_components=1, prefix='pca_amt'), ['e', 'monto']),
        ('passthrough_raw',     'passthrough',                                         passthrough_raw),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

In [5]:
# --- 1. Time-based split on raw data ---
raw_feature_cols = [c for c in df_raw.columns if c != 'fraude']

X_train_raw, X_test_raw, y_train_xgb, y_test_xgb = time_based_split(
    df=df_raw,
    features=raw_feature_cols,
    target='fraude',
    test_size=0.2,
)

# --- 2. Fit on train only, transform both (no leakage) ---
X_train_xgb = preprocessor_xgb.fit_transform(X_train_raw)
X_test_xgb  = preprocessor_xgb.transform(X_test_raw)

feature_names_xgb = preprocessor_xgb.get_feature_names_out()

df_train_xgb = pd.DataFrame(X_train_xgb, columns=feature_names_xgb, index=X_train_raw.index)[XGB_FEATURES]
df_test_xgb  = pd.DataFrame(X_test_xgb,  columns=feature_names_xgb, index=X_test_raw.index)[XGB_FEATURES]

print(f"Train shape: {df_train_xgb.shape}")
print(f"Test  shape: {df_test_xgb.shape}")
display(df_train_xgb.head())

Time-based split (no validation):
  Train : (120000, 18)
  Test  : (30000, 18)
Train shape: (120000, 20)
Test  shape: (30000, 20)


,a,b,d,e,f,g,l,m,n,o,p,monto,score,o_is_missing,f_is_zero,m_is_zero,monto_top1_flag,j_rare_grouped,pca_beh_1,pca_amt_1
83947,4.0,0.7388,14.0,0.139279,24.0,6.0,2361.0,442.0,1.0,1.0,1.0,22.18,25.0,1.0,0.0,0.0,0.0,0.958742,142.087290,-21.608309
9971,4.0,0.7548,20.0,0.514815,7.0,6.0,2324.0,73.0,1.0,1.0,0.0,6.00,7.0,1.0,0.0,0.0,0.0,0.958742,-226.439240,-37.788621
12230,4.0,0.9026,50.0,0.274167,1.0,6.0,235.0,232.0,1.0,0.0,1.0,26.67,91.0,0.0,0.0,0.0,0.0,0.958742,-66.439581,-17.118425
85763,4.0,0.8285,1.0,0.000000,4.0,6.0,658.0,0.0,1.0,0.0,0.0,40.01,91.0,0.0,0.0,1.0,0.0,0.958742,-300.091262,-3.778198
118081,2.0,0.5992,2.0,0.000000,264.0,2.0,2400.0,10.0,1.0,1.0,0.0,6.25,93.0,1.0,0.0,0.0,0.0,0.958742,-290.061147,-37.538186


## XGBoost

**Why XGBoost**

* We use a **boosted-tree model** because boosting turns weak learners into a strong classifier and focuses training on harder (misclassified) examples; XGBoost is a widely used GBM variant for classification/ranking tasks. 
* In fraud settings, reaction time and prioritization by **value and risk** matter, and false positives can create customer friction—so we need a model that ranks risk well and supports thresholding to control FP/FN trade-offs.  

**Practical notes for tree-based models (XGBoost)**

* **What trees do well:** they learn decision rules through splits, so they naturally capture **non-linear effects** and **feature interactions** without needing many manual transformations. This is useful in fraud, where risk often comes from combinations of signals rather than a single linear effect.
* **Less preprocessing required:** tree models typically don’t need feature scaling and are usually **less sensitive to skewed distributions** than linear models.
* **Missing values:** many boosted-tree implementations can handle **NaN** by learning a default direction in splits, which makes them robust to incomplete production data (although monitoring missingness rates is still important).

**What can be worse / risks in practice**

* **Overfitting risk:** boosted trees can overfit if depth/trees are too large or if there is leakage; regularization and early stopping are important.
* **Probability calibration:** the ranking can be strong, but predicted probabilities may be poorly calibrated (especially with class weighting), so threshold selection and calibration checks can matter.
* **Production drift:** fraud patterns and data distributions change over time; even strong models can degrade, so we should monitor drift and performance and retrain when needed.


**How we evaluate (aligned with the task goal)**

* Primary selection is based on **profit/loss optimization** (threshold chosen to maximize profit on validation), since fraud systems must balance fraud losses vs. opportunity cost from blocking legitimate transactions. 

In [6]:
import xgboost as xgb
from sklearn.pipeline import Pipeline

scale_pos_weight = y_train_xgb.value_counts()[0] / y_train_xgb.value_counts()[1]
print(f"Scale Pos Weight: {scale_pos_weight:.2f}")

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_xgb),
    ('classifier', xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='aucpr',
        scale_pos_weight=scale_pos_weight,
        tree_method='hist',
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    ))
])

# pipeline.fit preprocesses X_train_raw internally before passing to classifier
# eval_set must be pre-transformed (bypasses the pipeline steps)
model_pipeline.fit(
    X_train_raw, y_train_xgb,
    classifier__eval_set=[(preprocessor_xgb.transform(X_test_raw), y_test_xgb)],
    classifier__verbose=False,
)

y_pred_proba = model_pipeline.predict_proba(X_test_raw)[:, 1]

Scale Pos Weight: 18.33


# Evaluation

In [7]:
# ── XGBoost — full evaluation with the new modular API ───────────────────────

# 1. Score once — everything below only uses y_scores
y_scores_xgb = get_scores(model_pipeline, X_test_raw)


# 2. Compute all stats (no side effects)
pr_stats     = compute_pr_stats(y_true=y_test_xgb, y_scores=y_scores_xgb)
profit_stats = compute_profit_curve(y_true=y_test_xgb, y_scores=y_scores_xgb, monto=X_test_raw['monto'])
profit_stats_no_monto = compute_profit_curve(y_true=y_test_xgb, y_scores=y_scores_xgb, monto=None)


cost_FN, cost_FP = 1.00, 0.25  
beta = np.sqrt(cost_FN / cost_FP)  # β=2 for 4:1 cost ratio
fbeta_stats  = compute_fbeta_curve(y_true=y_test_xgb, y_scores=y_scores_xgb, beta=beta)

# Collect the three candidate thresholds for cross-referencing
t_profit = profit_stats['best_profit_threshold']
t_f1     = pr_stats['best_f1_threshold']
t_fb     = fbeta_stats['best_fbeta_threshold']

print(f"Profit-block-all  t = {0:.4f}  →  profit = {profit_stats['profits'][0]:,.2f}")
print(f"Profit-block-none  t = {1:.4f}  →  profit = {profit_stats['profits'][-1]:,.2f}")
print(f"Profit-optimal  t = {t_profit:.4f}  →  profit = {profit_stats['best_profit']:,.2f}")
print(f"Diff profit vs block-none: {profit_stats['best_profit'] - profit_stats['profits'][-1]:,.2f}")

print(f"\nF1-optimal      t = {t_f1:.4f}  →  F1     = {pr_stats['best_f1']:.4f}")
print(f"F2-optimal      t = {t_fb:.4f}  →  F2     = {fbeta_stats['best_fbeta']:.4f}")
print(f"AUC-PR = {pr_stats['auc_pr']:.4f}   ROC-AUC = {pr_stats['roc_auc']:.4f}")

cr_stats = compute_classification_report(
    y_true=y_test_xgb, y_scores=y_scores_xgb, threshold=t_profit
)
print_classification_report(cr_stats, model_name="XGBoost")



Profit-block-all  t = 0.0000  →  profit = -293,509.88
Profit-block-none  t = 1.0000  →  profit = -99,824.38
Profit-optimal  t = 0.7893  →  profit = -59,426.09
Diff profit vs block-none: 40,398.29

F1-optimal      t = 0.8356  →  F1     = 0.4320
F2-optimal      t = 0.6421  →  F2     = 0.4856
AUC-PR = 0.4159   ROC-AUC = 0.8819

XGBoost — Classification Report  (threshold = 0.7893)
──────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

           0     0.9765    0.9626    0.9695     28708
           1     0.3690    0.4861    0.4195      1292

    accuracy                         0.9421     30000
   macro avg     0.6728    0.7243    0.6945     30000
weighted avg     0.9504    0.9421    0.9458     30000



In [8]:
# 3. Plot PR curve (marks best-F1 point automatically)
plot_pr_curve(pr_stats, 
              model_name="XGBoost",
              highlight_thresholds=[(t_f1, 'F1-opt'), (t_fb, 'FB-opt')],)

# 4. Plot profit curve — overlay F1 and F2 reference thresholds
plot_profit_curve(
    profit_stats,
    model_name="Profit (with Monto) - XGBoost",
    highlight_thresholds=[(t_f1, 'F1-opt'), (t_fb, 'FB-opt')],
)

# 4. Plot profit curve — overlay F1 and F2 reference thresholds
plot_profit_curve(
    profit_stats_no_monto,
    model_name="Profit (No monto) - XGBoost",
    highlight_thresholds=[(t_f1, 'F1-opt'), (t_fb, 'FB-opt')],
)

# 5. Confusion matrix at the profit-optimal threshold
cm_stats = compute_confusion_matrix_data(y_true=y_test_xgb, y_scores=y_scores_xgb, threshold=t_profit)
plot_confusion_matrix(cm_stats, model_name="XGBoost")

# 6. F2 curve — overlay profit-optimal and F1 thresholds for comparison
plot_fbeta_by_threshold(
    fbeta_stats,
    model_name="XGBoost",
    highlight_thresholds=[(t_profit, 'Profit-opt'), (t_f1, 'F1-opt')],
)




  TP=628  FP=1074  FN=664  TN=27634  |  Precision=0.3690  Recall=0.4861


Best F2.0 threshold : 0.6421  →  F2.0 = 0.4856


# XGBoost — Model Conclusion

The XGBoost model demonstrates strong discrimination ability (ROC-AUC = 0.88) and delivers
substantial financial value under the business cost structure (25% margin loss on FP, 100%
fraud loss on FN).

### Summary

At the profit-optimal threshold (t = 0.789), the model:

- **Recovers ~80% of baseline losses**: reducing expected fraud loss from -$99,824.38 (no fraud detected)
 down to -$59,426 — a recovery of `~$40,398.29` on the test set.
- **Catches 48.6% of fraud cases** (628 / 1,292) while only blocking 3.7% of legitimate
  transactions (1,074 / 28,708), keeping customer friction low.
- **Precision on fraud (36.9%)** reflects the inherent difficulty of the task at a 4.3% base
  rate, but is acceptable given that the cost structure heavily favors catching high-value
  frauds over minimizing false alarms.
- **Key insight**: “profit-optimal” does not mean profitable; it means least bad given your function and model.

### Strengths

- The model's **ranking quality (ROC-AUC = 0.88)** is well above a naive baseline, suggesting
  the engineered features (PCA components, frequency encoding, missingness flags) carry genuine
  predictive signal.
- The **profit curve is smooth and unimodal**, indicating stable behavior across thresholds
  and that the chosen threshold is robust rather than a statistical artifact.
- The selected threshold (0.789) is consistent with the asymmetric cost structure (FN is substantially more expensive than FP), and aligns with a recall/precision trade-off that prioritizes limiting false blocks while still capturing meaningful fraud volume.

### Limitations & Risks

- **Recall gap (51.4% missed fraud):** Over half of fraud cases are not flagged at the
  profit-optimal threshold. If the business can absorb more FPs (e.g., through a softer
  block action like manual review), lowering the threshold toward the F2-optimal (0.642)
  would catch significantly more fraud.
- **AUC-PR = 0.42** leaves room for improvement. The model could benefit from better
  calibration, additional feature engineering on transaction velocity/network features,
  or ensemble stacking with a complementary algorithm.
- **Temporal drift:** The model was trained and evaluated on a static time-based split.
  Fraud patterns evolve; a production deployment requires periodic retraining and
  monitoring of score distributions over time.

### Recommendation

Deploy with threshold **t ≈ 0.79** for fully automated blocking. If a manual review queue
is feasible, consider a **two-tier policy**: auto-block above 0.79, queue for review
between 0.64 and 0.79 — this would recover additional fraud with controlled operational cost.
A baseline **Logistic Regression** comparison is recommended to confirm the gain from the
non-linear model is justified.

# Suggestions to Next Steps

Here, follow suggestions for the next development to improve the results.

### 1. Probability Calibration
XGBoost with `scale_pos_weight` tends to produce **poorly calibrated probabilities** — the raw score may not reflect true fraud probability. This matters because our threshold selection (t = 0.789) assumes the score is a reliable ranking signal. Applying **Platt Scaling** (`CalibratedClassifierCV(method='sigmoid')`) or **Isotonic Regression** post-training can:
- Make the probability outputs more interpretable and trustworthy
- Improve threshold stability — a calibrated model is less sensitive to small threshold shifts
- Enable better downstream decisions (e.g., tiered review queues based on probability buckets)

### 2. Cross-Validation
The current evaluation uses a single time-based train/test split (80/20). This means our metrics (AUC-PR = 0.42, ROC-AUC = 0.88) may be sensitive to **which 20% of time we happened to hold out**. Adding **time-series cross-validation** (`TimeSeriesSplit`) would:
- Provide confidence intervals around all reported metrics (mean ± std across folds)
- Detect whether model performance degrades on more recent data windows (early sign of temporal drift)
- Give a more robust basis for comparing XGBoost vs Logistic Regression

### 3. Feature Importance Analysis
XGBoost exposes multiple importance measures — **gain** (reduction in loss), **weight** (frequency of splits), and **cover** (number of samples affected). Analyzing these would:
- Confirm which engineered features (PCA components, frequency encoding, flags) actually drive splits
- Identify low-contribution features that could be removed to reduce overfitting risk
- Guide the feature set for the Logistic Regression baseline (simpler models need fewer, cleaner features)
- Provide interpretability for stakeholders (regulators often require explainable fraud models)

### 4. Hyperparameter Tuning
The current model uses default XGBoost parameters except for `scale_pos_weight` and `early_stopping_rounds`. A targeted search over the most impactful parameters would likely improve profit recovery:

| Parameter | Effect |
|---|---|
| `max_depth` (3–8) | Controls tree complexity; lower = less overfit |
| `learning_rate` (0.01–0.3) | Step size; lower with more trees often better |
| `subsample` / `colsample_bytree` | Stochastic regularization |
| `min_child_weight` | Minimum sample weight in leaf; helps with imbalance |

Use `optuna` or `BayesSearchCV` with **profit as the scoring function** (not AUC) to ensure tuning aligns with the business objective.

### 5. Logistic Regression Baseline
Before accepting XGBoost's complexity, we should verify the non-linear model actually outperforms a linear one. Logistic Regression with `class_weight='balanced'` and `StandardScaler` preprocessing serves as:
- A **sanity check**: if LR achieves similar profit, XGBoost's overhead (training time, explainability, maintenance) is not justified
- A **reference point** for the AUC-PR and ROC-AUC gap
- A naturally **calibrated model** — LR probabilities are well-calibrated by default, making threshold selection more reliable

### 6. Production Monitoring Plan
A model trained on historical data will eventually degrade as fraud patterns evolve. The minimum monitoring setup should track:

| Signal | Metric | Alert When |
|---|---|---|
| Score distribution shift | KS statistic (train vs live) | > 0.1 |
| Fraud rate drift | Rolling 7-day fraud rate | ±50% vs baseline |
| Profit degradation | Rolling weekly profit | < 70% of initial |
| FP rate increase | Blocked legit % | > 5% |
